In [0]:
# Databricks notebook source
import unittest
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, when, sha2, concat_ws

spark = SparkSession.builder.getOrCreate()

# --- CENTRALIZED CONFIGURATION ---
catalog = "ecommerce_analytics_dev"
bronze_table = f"{catalog}.bronze_layer.events_raw"
silver_table = f"{catalog}.silver_layer.events_cleaned"
gold_table   = f"{catalog}.gold_layer.fact_sales"

In [0]:
def calculate_row_integrity_hash(table_name):
    """Implements SHA-256 hashing to verify row integrity."""
    df = spark.table(table_name)
    hash_df = df.withColumn("row_hash", sha2(concat_ws("||", *df.columns), 256))
    return hash_df.filter(col("row_hash").isNull()).count()

In [0]:
class EcommerceDataQualityAudit(unittest.TestCase):

    # --- 1. BRONZE: AUDIT VALIDATION ---
    def test_bronze_audit_trail(self):
        """Domain 4.1: Ensure ingestion metadata is present for every row."""
        if spark.catalog.tableExists(bronze_table):
            df = spark.table(bronze_table)
            missing_metadata = df.filter(col("source_file").isNull() | col("ingestion_timestamp").isNull()).count()
            self.assertEqual(missing_metadata, 0, f"Found {missing_metadata} rows missing audit metadata.")
        else:
            self.skipTest("Bronze table not found.")

    # --- 2. SILVER: BUSINESS RULE VALIDATION ---
    def test_silver_cleaning_logic(self):
        """Domain 4.2: Ensure price > 0 quarantine rules applied."""
        if spark.catalog.tableExists(silver_table):
            invalid_records = spark.table(silver_table).filter(col("price") <= 0).count()
            self.assertEqual(invalid_records, 0, f"Failure: {invalid_records} invalid prices in Silver.")
        else:
            self.skipTest("Silver table not found.")

    # --- 3. GOLD: STAR SCHEMA INTEGRITY ---
    def test_gold_fact_uniqueness(self):
        """Domain 4.3: Ensure Fact table maintains unique transaction grains."""
        if spark.catalog.tableExists(gold_table):
            df = spark.table(gold_table)
            is_unique = df.count() == df.distinct().count()
            self.assertTrue(is_unique, "Duplicates detected in Gold Fact Sales.")
        else:
            self.skipTest("Gold table not found.")

    # --- 4. ROW LEVEL INTEGRITY (SHA CHECK) ---
    def test_row_integrity_sha(self):
        """Verify data integrity using deterministic SHA-256 hashes."""
        if spark.catalog.tableExists(silver_table):
            null_hashes = calculate_row_integrity_hash(silver_table)
            self.assertEqual(null_hashes, 0, f"SHA Integrity Failed: {null_hashes} invalid hashes.")

    # --- 5. RECONCILIATION AUDIT (ROW INTEGRITY) ---
    def test_row_reconciliation(self):
        """Domain 3.4: Verify row counts reconcile across layers"""
        if spark.catalog.tableExists(bronze_table) and spark.catalog.tableExists(silver_table):
            bronze_count = spark.table(bronze_table).count()
            silver_count = spark.table(silver_table).count()
            
            # Genuine Insight: Print for the logs
            print(f"\n📊 RECONCILIATION INSIGHT:")
            print(f"   Bronze Rows: {bronze_count:,}")
            print(f"   Silver Rows: {silver_count:,}")
            print(f"   Dropped Records: {bronze_count - silver_count:,}")
            
            self.assertLessEqual(silver_count, bronze_count, "ERROR: Silver has more rows than Bronze!")
        else:
            self.skipTest("Tables missing for reconciliation check.")

In [0]:
# --- EXECUTION ENGINE ---
print("🚀 INITIALIZING COMPREHENSIVE DATA QUALITY AUDIT...")
suite = unittest.TestLoader().loadTestsFromTestCase(EcommerceDataQualityAudit)
result = unittest.TextTestRunner(verbosity=1).run(suite)

# --- FINAL PROJECT SUBMISSION REPORT ---
print("\n" + "="*55)
print(" ⭐ PROJECT SUBMISSION REPORT: DATA QUALITY AUDIT ⭐")
print("="*55)
print(f"Total Tests Executed: {result.testsRun}")
print(f"Tests Passed:         {result.testsRun - len(result.failures) - len(result.errors)}")
print(f"Tests Skipped:        {len(result.skipped)}")
print(f"Tests Failed:         {len(result.failures) + len(result.errors)}")

if result.wasSuccessful():
    print("\n✅ STATUS: READY FOR PRODUCTION DEPLOYMENT")
else:
    print("\n❌ STATUS: PIPELINE BLOCKED - FIX QUALITY ERRORS")
    # Halt the Job if any test fails
    raise Exception("Automation Halted: Comprehensive Data Quality Validation Failed.")
print("="*55)